# SHL: 29 audio and grammar channels in one file

1. On Vast, open this notebook in the SHL Training kernel. For Kaggle, import this notebook. Attach **SHL Hiring Assessment 2026**, select a **GPU**, enable **Internet**, and choose **Run All**. No API keys or paid APIs are required inside the notebook. Public model weights and dependencies download automatically.
2. Existing transcripts are required: attach outputs of `prabalsinghx/shl-2026-crisperwhisper-verbatim-transcripts`. Hash-matching verbatim transcripts are reused. On Vast, existing Crisper and original transcripts are loaded automatically. No ASR inference or ASR model download occurs.
3. Output: **`/kaggle/working/shl_all_channels.h5`**. This is the only generated data artifact: tensors, existing transcripts, grammar edits, timestamps, source metadata, model revisions, and extraction checkpoints are inside it. Models and generated helper code live under `/tmp`, outside saved notebook outputs. On Vast the output is `/workspace/shl/audio_embeddings/shl_all_channels.h5`. Original WAVs stay in competition inputs.
4. This is frozen embedding inference and feature extraction using existing transcripts. It does **not** train a grammar scorer, PCA, a stacker, or attention pooling. The trainable chunk aggregator belongs in your later training script. ASR/grammar/repair predictions are estimates and may be wrong.
5. Resource controls are editable in CONFIG. Effective CPU capacity respects container quotas, not the host CPU count. Existing stage checkpoints survive this performance update; faster speech inference uses recorded bfloat16/float16 autocast with float32 vectors.
6. All **769 training + 216 test** recordings preserve source CSV order. Original training labels, including zero labels, are unchanged; random test labels are ignored and replaced with NaN.
7. It prints stage progress. A full run is substantial and may exceed a Kaggle session. Existing completed recording/stage checkpoints are reused. To resume in a new session, attach the previous `.h5` as a dataset and set `RESUME_FILE` below. The file records `complete=False` until all stages finish for all 985 WAVs.
8. Pretrained assets and existing transcript exports have their own licences; model IDs and pinned revisions are recorded. CrisperWhisper outputs and the chosen GEC model have noncommercial licence terms. Check applicability to your competition use.

## One recording's shapes

1. Fixed vector **`X[i]`: `(9888,)`**, float32, with a same-shaped boolean `available[i]` mask. There are **26 fixed channels**, with named slices recorded in the file.
2. WavLM layers 4/8/12 × mean/std: **4608 values**, included in `X` (six 768D channels). HuBERT and wav2vec2 each contribute layer-4 mean, layer-12 mean, and layer-12 std: **2304 values each**.
3. Eight acoustic groups: log-mel **128**, MFCC/delta/delta-delta **26 each**, pitch/energy/pause/word-duration statistics **8 each**: **238 values**.
4. Existing-transcript semantic embedding **384** plus language-model surprisal statistics **8**: **392 values**. Surprisal is not an error count or calibrated proficiency score.
5. Grammar accuracy **16**, syntax complexity **12**, repair behaviour **8**, grammatical hesitation timing **6**: **42 values**. Counts/rates and clause measures are estimated from existing transcripts. Qwen repair analysis is disabled: all 8 repair values and pause-before-repair remain NaN with availability=False. Undefined ratios remain missing.
6. Three additional sequence channels in `records/0000/`, etc.: **`word_wavlm`: `(W,768)`**, **`phoneme_probabilities`: `(F,392)`**, **`chunk_wavlm`: `(C,768)`**. W is aligned word count; F is phoneme-model frame count; C is the number of 10-second chunks, with tiny final tails merged. Word and chunk embeddings use WavLM layer 8. Phoneme probabilities include CTC blank/special tokens, use float16, and have their own timestamps and vocabulary.
7. Existing transcripts are used unchanged: **no Qwen cleanup, no transcription**. Historical `fluent` dataset/channel names remain for compatibility but contain/use verbatim text. A separate GEC proposal supplies grammar-edit features without changing the stored transcript. Token mappings are identity mappings.
8. **No pretrained `(768,)` attention result is claimed.** Train masked attention over `chunk_wavlm` using your training folds later. Whole-recording means/std use all unpadded encoder frames, weighted by frame count. Chunk context resets every 10 seconds.

In [ ]:
from pathlib import Path
import os

# Run All processes the complete dataset. LIMIT is only for a user-selected smoke run.
ON_KAGGLE = Path("/kaggle/input").is_dir()
CONFIG = {
    "DATA_ROOT": None if ON_KAGGLE else "/workspace/shl/data/audio_full",
    "OUTPUT": "/kaggle/working/shl_all_channels.h5" if ON_KAGGLE else "/workspace/shl/audio_embeddings/shl_all_channels.h5",
    "RESUME_FILE": None,  # e.g. /kaggle/input/my-previous-output/shl_all_channels.h5
    "CRISPER_TRANSCRIPT_ROOT": None if ON_KAGGLE else "/workspace/shl/data/text_input_crisper",
    "ORIGINAL_TRANSCRIPT_ROOT": None if ON_KAGGLE else "/workspace/shl/data/text_input",
    "CPU_THREADS": None,  # Auto: up to 16 effective CPU threads.
    "CPU_WORKERS": None,  # Auto: up to 32 independent CPU DSP workers.
    "GPU_BATCH_SIZE": 16,  # Equal-length audio chunks; CUDA OOM automatically splits the batch.
    "LIMIT": None,  # None = all 985. Setting an integer produces a clearly marked partial file.
}
os.environ["HF_HOME"] = "/tmp/shl_channel_model_cache" if ON_KAGGLE else "/opt/shl-cache/huggingface"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = str(min(16, os.cpu_count() or 1))
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMBA_NUM_THREADS"] = "1"
print("Output:", CONFIG["OUTPUT"])

## Dependencies

1. This cell installs extraction dependencies and the English parser automatically. It leaves Kaggle's PyTorch/CUDA installation in place. WhisperX is installed without its unrelated ASR/diarization dependency stack because only its forced aligner is used.
2. If the GPU/internet checks fail, enable those Kaggle settings and restart the session. If Kaggle already imported an incompatible dependency before this cell, restart the session and Run All from the first cell.

In [ ]:
import sys, subprocess
assert sys.version_info >= (3, 10), "Use a current Kaggle Python image."
subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "transformers==4.57.6", "huggingface_hub>=0.34,<1", "accelerate>=0.26,<2",
    "spacy>=3.8,<3.9", "errant>=3,<4",
    "h5py>=3.11", "soundfile>=0.12", "librosa>=0.10,<0.12", "webrtcvad-wheels",
    "sentencepiece", "protobuf", "soxr", "nltk>=3.9", "tqdm", "pandas>=2",
    "scipy>=1.11", "threadpoolctl"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "whisperx==3.8.6"])
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
print("GPU:", torch.cuda.get_device_name(0), "| Torch:", torch.__version__)

## Embedded extractor

1. Everything required for extraction is embedded here; no repository checkout is needed.
2. Helper code is materialized under `/tmp`. This cell defines the extractor without running it.

In [ ]:
RUNTIME_SOURCE = '"""Self-contained, inference-only extraction into one resumable HDF5 file."""\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nimport time\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport h5py\nimport numpy as np\nimport pandas as pd\nimport soundfile as sf\nimport torch\nfrom scipy.signal import resample_poly\nfrom tqdm.auto import tqdm\n\nACCURACY = [\'error_rate\', \'severe_error_rate\', \'error_free_clause_ratio\',\n \'error_free_sentence_ratio\', \'tense_error_rate\', \'agreement_error_rate\',\n \'verb_form_error_rate\', \'article_error_rate\', \'preposition_error_rate\',\n \'pronoun_error_rate\', \'plural_error_rate\', \'word_order_error_rate\',\n \'missing_word_rate\', \'extra_word_rate\', \'negation_error_rate\', \'multi_error_clause_ratio\']\nSYNTAX = [\'mean_clause_length\', \'clauses_per_sentence\', \'subordinate_clause_ratio\',\n \'relative_clause_rate\', \'conditional_rate\', \'passive_construction_rate\',\n \'complex_sentence_ratio\', \'mean_dependency_depth\', \'syntactic_depth_p90\',\n \'complete_clause_ratio\', \'fragment_rate\', \'correct_complex_clause_ratio\']\nREPAIRS = [\'repair_attempt_rate\', \'successful_repair_rate\', \'failed_repair_rate\',\n \'repair_success_ratio\', \'false_start_rate\', \'abandoned_clause_rate\',\n \'candidate_disfluency_repetition_rate\', \'restart_completion_ratio\']\nTIMING = [\'within_clause_pause_rate\', \'boundary_pause_fraction\',\n \'pause_before_repair_median\', \'long_within_clause_pause_rate\',\n \'mean_words_per_fluent_run\', \'local_speech_rate_cv\']\nMODELS = {\n \'asr\': \'nyralabs/CrisperWhisper2.0_large\',\n \'gec\': \'vennify/t5-base-grammar-correction\',\n \'semantic\': \'sentence-transformers/all-MiniLM-L6-v2\',\n \'surprisal\': \'distilbert/distilgpt2\',\n \'wavlm\': \'microsoft/wavlm-base-plus\',\n \'hubert\': \'facebook/hubert-base-ls960\',\n \'wav2vec2\': \'facebook/wav2vec2-base-960h\',\n \'phoneme\': \'facebook/wav2vec2-lv-60-espeak-cv-ft\',\n \'align\': \'facebook/wav2vec2-base-960h\',\n}\nSTAGES = [\'asr\', \'fluent\', \'align\', \'grammar\', \'text\', \'dsp\',\n          \'wavlm\', \'hubert\', \'wav2vec2\', \'phoneme\']\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'): h.update(block)\n    return h.hexdigest()\n\n\ndef words(text): return re.findall(r"[A-Za-z]+(?:\'[A-Za-z]+)?", text)\ndef ratio(a, b): return float(a / b) if b else np.nan\ndef rate(a, b): return 100 * ratio(a, b)\ndef stats(x):\n    x = np.asarray(x, dtype=float); x = x[np.isfinite(x)]\n    if not len(x): return np.full(8, np.nan, np.float32)\n    return np.asarray([x.mean(), x.std(), x.min(), x.max(),\n        *np.quantile(x, [.1, .5, .9]), ratio(x.std(), abs(x.mean()))], np.float32)\n\n\ndef resolve_audio(folder, filenames):\n    """Resolve only CSV source audio; Jupyter preview checkpoints are not inputs."""\n    folder = Path(folder); expected = set(filenames); lookup = {}; extras = []\n    for path in folder.rglob(\'*.wav\'):\n        if \'.ipynb_checkpoints\' in path.relative_to(folder).parts:\n            continue\n        if path.name not in expected:\n            extras.append(path.name)\n            continue\n        if path.name in lookup:\n            raise ValueError(f\'Duplicate source WAV {path.name}: {lookup[path.name]} and {path}\')\n        lookup[path.name] = path\n    missing = sorted(expected - set(lookup))\n    if missing:\n        raise ValueError(f\'{folder.name}: {len(missing)} required WAVs missing; examples: {missing[:10]}\')\n    if extras:\n        print(f\'{folder.name}: ignored {len(extras)} WAVs not listed in source CSV: {sorted(extras)[:10]}\', flush=True)\n    return lookup\n\n\ndef schema():\n    definitions = []\n    for model in [\'wavlm\', \'hubert\', \'wav2vec2\']:\n        views = [(l, p) for l in [4, 8, 12] for p in [\'mean\', \'std\']] if model == \'wavlm\' else [(4, \'mean\'), (12, \'mean\'), (12, \'std\')]\n        for layer, pooling in views:\n            definitions.append((f\'{model}_layer{layer}_{pooling}\', 768, None))\n    definitions.extend([(name, dim, None) for name, dim in [\n        (\'logmel\', 128), (\'mfcc\', 26), (\'mfcc_delta\', 26), (\'mfcc_delta2\', 26),\n        (\'pitch\', 8), (\'energy\', 8), (\'pauses\', 8), (\'word_timing\', 8),\n        (\'fluent_semantic\', 384), (\'fluent_surprisal\', 8)]])\n    definitions.extend([(n, len(cols), cols) for n, cols in [\n        (\'grammar_accuracy\', ACCURACY), (\'syntax_complexity\', SYNTAX),\n        (\'repair_behaviour\', REPAIRS), (\'grammar_timing\', TIMING)]])\n    offset = 0; result = {}\n    for name, width, columns in definitions:\n        result[name] = dict(start=offset, stop=offset + width, dimensions=width,\n            columns=columns or [f\'{name}_{i}\' for i in range(width)])\n        offset += width\n    assert offset == 9888 and len(result) == 26\n    return result\n\n\ndef cpu_capacity():\n    count=len(os.sched_getaffinity(0)) if hasattr(os,\'sched_getaffinity\') else (os.cpu_count() or 1)\n    for path in [\'/sys/fs/cgroup/cpu.max\',\'/sys/fs/cgroup/cpu/cpu.cfs_quota_us\']:\n        try:\n            if path.endswith(\'cpu.max\'):\n                quota,period=Path(path).read_text().split()\n                if quota!=\'max\':count=min(count,max(1,math.ceil(int(quota)/int(period))))\n            else:\n                quota=int(Path(path).read_text());period=int(Path(path).with_name(\'cpu.cfs_period_us\').read_text())\n                if quota>0:count=min(count,max(1,math.ceil(quota/period)))\n        except (OSError,ValueError):pass\n    return count\n\n\ndef gpu_batch_map(items,batch_size,fn):\n    """Yield results in input order; split CUDA OOM batches without dropping inputs."""\n    def attempt(batch):\n        try:\n            values=fn(batch)\n            if len(values)!=len(batch):raise ValueError(\'Batch output count differs from inputs.\')\n            return values\n        except torch.cuda.OutOfMemoryError:\n            if len(batch)==1:raise\n        # Leave the exception handler first so failed-forward tensors can be freed.\n        gc.collect();torch.cuda.empty_cache()\n        mid=len(batch)//2\n        return attempt(batch[:mid])+attempt(batch[mid:])\n    for start in range(0,len(items),batch_size):\n        batch=items[start:start+batch_size]\n        yield from zip(batch,attempt(batch))\n\n\ndef complete_json(text):\n    """Read the first complete object, ignoring fences or trailing model commentary."""\n    start=text.find(\'{\')\n    if start<0:return None\n    try:value,_=json.JSONDecoder().raw_decode(text[start:])\n    except ValueError:return None\n    return value if isinstance(value,dict) else None\n\n\ndef dsp_worker(task):\n    """CPU-only worker. Parent owns the HDF5 writer; workers never open its file."""\n    i,path,aligned=task\n    torch.set_num_threads(1)\n    from threadpoolctl import threadpool_limits\n    threadpool_limits(limits=1)\n    import librosa,webrtcvad\n    x,sr=sf.read(path,dtype=\'float32\',always_2d=True);x=x.mean(axis=1)\n    if sr!=16000:\n        d=math.gcd(sr,16000);x=resample_poly(x,16000//d,sr//d).astype(\'f4\')\n    sr=16000\n    power=np.abs(librosa.stft(x,n_fft=512,hop_length=160,win_length=400))**2\n    mel=librosa.feature.melspectrogram(S=power,sr=sr,n_mels=64)\n    logmel=librosa.power_to_db(mel,ref=1.,top_db=None)\n    mfcc=librosa.feature.mfcc(S=logmel,n_mfcc=13)\n    pool=lambda a:np.r_[a.mean(axis=1),a.std(axis=1)]\n    vectors={\'logmel\':pool(logmel),\'mfcc\':pool(mfcc)}\n    width=min(9,mfcc.shape[1] if mfcc.shape[1]%2 else mfcc.shape[1]-1)\n    for order,name in [(1,\'mfcc_delta\'),(2,\'mfcc_delta2\')]:\n        vectors[name]=pool(librosa.feature.delta(mfcc,order=order,width=width)) if width>=3 else np.full(26,np.nan)\n    rms=librosa.feature.rms(y=x,frame_length=400,hop_length=160)[0]\n    vectors[\'energy\']=stats(20*np.log10(np.maximum(rms,1e-10)))\n    f0,flag,prob=librosa.pyin(x,sr=sr,fmin=65,fmax=500,frame_length=1024,hop_length=640)\n    selected=f0[flag & (prob>=.7) & np.isfinite(f0)]\n    vectors[\'pitch\']=stats(12*np.log2(selected/np.median(selected))) if len(selected) else np.full(8,np.nan)\n    pcm=(np.clip(x,-1,1)*32767).astype(\'<i2\');vad=webrtcvad.Vad(2)\n    voiced=np.asarray([vad.is_speech(pcm[a:a+480].tobytes(),sr) for a in range(0,len(pcm)-479,480)])\n    true=np.flatnonzero(voiced);pauses=[]\n    if len(true):\n        interior=voiced[true[0]:true[-1]+1];edges=np.flatnonzero(np.diff(np.r_[True,interior,True]))\n        pauses=[(b-a)*.03 for a,b in zip(edges[::2],edges[1::2]) if (b-a)*.03>=.25]\n    vectors[\'pauses\']=stats(pauses) if pauses else np.array([0,0,0,0,0,0,0,np.nan])\n    durations=[w[\'end\']-w[\'start\'] for w in aligned if \'start\' in w and \'end\' in w and w[\'end\']>w[\'start\']]\n    vectors[\'word_timing\']=stats(durations)\n    metadata=dict(duration_s=len(x)/sr,speech_fraction=float(voiced.mean()) if len(voiced) else None,\n        pause_count=len(pauses),words_per_second=len(aligned)/(len(x)/sr))\n    return i,vectors,metadata\n\n\nclass Extractor:\n    def __init__(self, config):\n        self.config = config; self.device = \'cuda:0\'\n        if not torch.cuda.is_available(): raise RuntimeError(\'Enable a Kaggle GPU accelerator.\')\n        capacity=cpu_capacity()\n        self.cpu_threads=int(config.get(\'CPU_THREADS\') or min(16,capacity))\n        self.cpu_workers=int(config.get(\'CPU_WORKERS\') or min(32,capacity))\n        self.gpu_batch=int(config.get(\'GPU_BATCH_SIZE\',16))\n        self.text_batch=int(config.get(\'TEXT_BATCH_SIZE\',4))\n        if min(self.cpu_threads,self.cpu_workers,self.gpu_batch,self.text_batch)<1:raise ValueError(\'Resource settings must be positive.\')\n        torch.set_num_threads(self.cpu_threads); torch.manual_seed(42)\n        torch.backends.cuda.matmul.allow_tf32=True\n        torch.backends.cudnn.allow_tf32=True\n        print(f\'Resources: {capacity} available CPUs; {self.cpu_workers} DSP workers; {self.cpu_threads} CPU threads; GPU batch {self.gpu_batch}; text batch {self.text_batch}\',flush=True)\n        self.channels = schema(); self.jobs = self.discover()\n        if config.get(\'LIMIT\'): self.jobs = self.jobs[:config[\'LIMIT\']]\n        self.path = Path(config[\'OUTPUT\']); self.path.parent.mkdir(parents=True, exist_ok=True)\n        resume = config.get(\'RESUME_FILE\')\n        if resume and not self.path.exists():\n            import shutil\n            shutil.copy2(resume, self.path)\n        self.h = h5py.File(self.path, \'a\')\n        signature = hashlib.sha256(json.dumps(dict(channels=self.channels,\n            audio=[(r[\'split\'],r[\'filename\'],r[\'sha\'],r[\'source_row\'],r[\'label\'] if r[\'split\']==\'train\' else None) for r in self.jobs]),sort_keys=True).encode()).hexdigest()\n        if \'X\' in self.h:\n            # Migration for the original notebook: performance changes must not discard completed work.\n            if self.h[\'X\'].shape!=(len(self.jobs),9888) or json.loads(self.h.attrs[\'channels_json\'])!=self.channels:\n                raise ValueError(\'Existing HDF5 schema differs; use a new OUTPUT.\')\n            for key in [\'filename\',\'split\',\'sha\']:\n                if list(self.h[\'metadata\'][key].asstr()[:])!=[r[key] for r in self.jobs]:\n                    raise ValueError(\'Existing HDF5 source audio differs; use a new OUTPUT.\')\n            for key in [\'source_row\',\'label\']:\n                if not np.array_equal(self.h[\'metadata\'][key][:],np.asarray([r[key] for r in self.jobs]),equal_nan=True):\n                    raise ValueError(\'Existing HDF5 ordering/labels differ; use a new OUTPUT.\')\n            if \'data_signature\' in self.h.attrs and self.h.attrs[\'data_signature\']!=signature:\n                raise ValueError(\'Existing HDF5 input signature differs.\')\n            for key,model_id in MODELS.items():\n                if key in self.h[\'provenance\'] and self.h[\'provenance\'][key].attrs.get(\'model_id\')!=model_id:\n                    raise ValueError(\'Existing model provenance differs.\')\n            print(\'Resuming existing HDF5; completed recording/stage checkpoints preserved.\',flush=True)\n        if \'X\' not in self.h:\n            n = len(self.jobs)\n            self.h.attrs.update(signature=signature, complete=False, inference_only=True,\n                schema_version=1, fixed_channels=26, sequence_channels=3,\n                fixed_dimensions=9888, test_labels=\'unknown; random CSV placeholders ignored\',\n                attention_pooling=\'Not trained: save chunk inputs for downstream fold-local training\',\n                channels_json=json.dumps(self.channels), configuration_json=json.dumps(config),\n                source_code=Path(__file__).read_text(), limitations_json=json.dumps([\n                    \'ASR, grammatical corrections, repairs, syntax and word alignment are estimates.\',\n                    \'GEC edits include stylistic changes; no edit is asserted to be ground truth.\',\n                    \'Severity and clause-completeness are explicitly heuristic proxies.\',\n                    \'Qwen cleanup disabled; repair behaviour and pause-before-repair are unavailable.\',\n                    \'Phoneme posteriors are uncalibrated acoustic model probabilities, not grammar scores.\',\n                    \'Frozen mean/std views are correlated; predictive power is unvalidated.\',\n                    \'Fit all learned pooling, scalers, imputers, PCA and stackers inside training folds.\',\n                    \'Full inference can exceed one Kaggle session; resume from the same single HDF5.\',\n                ]))\n            self.h.create_dataset(\'X\', (n,9888), dtype=\'f4\', fillvalue=np.nan,\n                chunks=(1,9888), compression=\'gzip\')\n            self.h.create_dataset(\'available\', (n,9888), dtype=\'bool\', chunks=(1,9888), compression=\'gzip\')\n            meta = self.h.create_group(\'metadata\'); string = h5py.string_dtype(\'utf-8\')\n            for key in [\'filename\',\'split\',\'sha\']:\n                meta.create_dataset(key, data=np.asarray([r[key] for r in self.jobs],dtype=object), dtype=string)\n            for key in [\'source_row\',\'label\',\'duration\']:\n                meta.create_dataset(key, data=np.asarray([r[key] for r in self.jobs]))\n            meta.create_dataset(\'label_known\', data=[r[\'split\']==\'train\' for r in self.jobs])\n            meta.create_dataset(\'train_indices\', data=[i for i,r in enumerate(self.jobs) if r[\'split\']==\'train\'])\n            meta.create_dataset(\'test_indices\', data=[i for i,r in enumerate(self.jobs) if r[\'split\']==\'test\'])\n            source = self.h.create_group(\'source_tables\')\n            for name in [\'train.csv\', \'sample_submission.csv\']:\n                p = self.root/name\n                if p.exists(): source.create_dataset(name, data=p.read_text(), dtype=string)\n            source.create_dataset(\'test_filenames\', data=np.asarray(\n                pd.read_csv(self.root/\'test.csv\').filename.astype(str),dtype=object),dtype=string)\n            self.h.create_group(\'records\'); self.h.create_group(\'provenance\')\n        self.h.attrs[\'data_signature\']=signature\n        versions=self.h[\'provenance\'].require_group(\'code_versions\')\n        source=Path(__file__).read_text();code_sha=sha(__file__)\n        if code_sha not in versions:versions.create_dataset(code_sha,data=source,dtype=h5py.string_dtype(\'utf-8\'))\n        self.h.attrs[\'performance_configuration_json\']=json.dumps(dict(cpu_threads=self.cpu_threads,cpu_workers=self.cpu_workers,gpu_batch=self.gpu_batch,text_batch=self.text_batch))\n        self.h.attrs[\'complete\'] = False\n        self.h.flush()\n\n    def discover(self):\n        if self.config.get(\'DATA_ROOT\'):\n            candidates = [Path(self.config[\'DATA_ROOT\'])]\n        else:\n            candidates = [p.parent for p in Path(\'/kaggle/input\').rglob(\'train.csv\')\n                if (p.parent/\'test.csv\').exists() and (p.parent/\'train\').is_dir()\n                and (p.parent/\'test\').is_dir()]\n        if len(candidates)!=1: raise ValueError(\'Attach competition data or set DATA_ROOT explicitly.\')\n        self.root = candidates[0]; jobs=[]\n        for split, expected in [(\'train\',769),(\'test\',216)]:\n            table = pd.read_csv(self.root/f\'{split}.csv\')\n            if len(table)!=expected or not table.filename.is_unique: raise ValueError(\'Unexpected source counts/duplicates.\')\n            lookup=resolve_audio(self.root/split,table.filename)\n            for source_row,row in table.iterrows():\n                p=lookup[row.filename]; info=sf.info(p)\n                label=float(row.label) if split==\'train\' else np.nan\n                if split==\'train\' and (not np.isfinite(label) or not 0<=label<=5): raise ValueError(\'Invalid train label.\')\n                jobs.append(dict(split=split,filename=str(row.filename),source_row=int(source_row),\n                    label=label,path=str(p),sha=sha(p),duration=info.duration))\n        print(\'Validated all 769 train + 216 test WAVs; test labels ignored.\',flush=True)\n        return jobs\n\n    def record(self,i): return self.h[\'records\'].require_group(f\'{i:04d}\')\n    def done(self,i,stage): return bool(self.record(i).attrs.get(stage+\'_done\',False))\n    def put(self,g,name,value):\n        if name in g: del g[name]\n        if isinstance(value,str): g.create_dataset(name,data=value,dtype=h5py.string_dtype(\'utf-8\'))\n        else:\n            value=np.asarray(value)\n            g.create_dataset(name,data=value,compression=\'lzf\' if value.ndim and value.size else None)\n    def getjson(self,g,name,default=None):\n        if name not in g: return default\n        return json.loads(g[name].asstr()[()])\n    def fixed(self,i,name,value):\n        v=np.asarray(value,np.float32); spec=self.channels[name]\n        if v.shape!=(spec[\'dimensions\'],) or np.isinf(v).any(): raise ValueError(\'Invalid channel \'+name)\n        self.h[\'X\'][i,spec[\'start\']:spec[\'stop\']]=v\n        self.h[\'available\'][i,spec[\'start\']:spec[\'stop\']]=np.isfinite(v)\n    def pending(self,stage): return [i for i in range(len(self.jobs)) if not self.done(i,stage)]\n    def each(self,stage,indices,fn,progress=True):\n        for i in (tqdm(indices,desc=stage) if progress else indices):\n            g=self.record(i)\n            try:\n                fn(i,g); g.attrs[stage+\'_done\']=True\n                if stage+\'_error\' in g.attrs: del g.attrs[stage+\'_error\']\n            except Exception as e:\n                g.attrs[stage+\'_error\']=f\'{type(e).__name__}: {str(e)[:2000]}\'\n                tqdm.write(f"{stage} failed: {self.jobs[i][\'split\']}/{self.jobs[i][\'filename\']}: {e}")\n                if isinstance(e,torch.cuda.OutOfMemoryError): torch.cuda.empty_cache()\n            self.drain_dsp()\n            self.h.flush()\n    def audio(self,i):\n        p=self.jobs[i][\'path\']\n        if sha(p)!=self.jobs[i][\'sha\']: raise ValueError(\'Source WAV changed.\')\n        x,sr=sf.read(p,dtype=\'float32\',always_2d=True); x=x.mean(axis=1)\n        if sr!=16000:\n            d=math.gcd(sr,16000); x=resample_poly(x,16000//d,sr//d).astype(\'f4\')\n        if len(x)<400 or not np.isfinite(x).all(): raise ValueError(\'Invalid or too-short audio.\')\n        return x\n    def revision(self,key):\n        from huggingface_hub import HfApi\n        g=self.h[\'provenance\'].require_group(key)\n        if \'revision\' not in g.attrs:\n            revision = \'f4334f6e8193f2691212d49b20fa12d370e13896\' if key==\'asr\' else HfApi().model_info(MODELS[key]).sha\n            g.attrs.update(model_id=MODELS[key],revision=revision)\n        return str(g.attrs[\'revision\'])\n    def release(self,*objects):\n        # Callers also delete their own references before empty_cache.\n        gc.collect(); torch.cuda.empty_cache()\n\n    def transcribe(self):\n        """Historical stage name retained for checkpoint compatibility; no ASR runs."""\n        sources={};originals={}\n        root=Path(self.config.get(\'CRISPER_TRANSCRIPT_ROOT\') or \'/workspace/shl/data/text_input_crisper\')\n        for split in [\'train\',\'test\']:\n            path=root/f\'{split}_transcripts_crisper.csv\'\n            if path.exists():\n                table=pd.read_csv(path,keep_default_na=False)\n                if not table.filename.is_unique:raise ValueError(\'Duplicate Crisper transcript keys.\')\n                for r in table.to_dict(\'records\'):sources[(split,r[\'filename\'])]=(r,str(path))\n        for p in Path(\'/kaggle/input\').rglob(\'transcripts_crisper.jsonl\'):\n            for line in p.open():\n                if line.strip():\n                    r=json.loads(line);sources.setdefault((r.get(\'split\'),r.get(\'filename\')),(r,str(p)))\n        original_root=Path(self.config.get(\'ORIGINAL_TRANSCRIPT_ROOT\') or \'/workspace/shl/data/text_input\')\n        for split in [\'train\',\'test\']:\n            p=original_root/f\'{split}.csv\'\n            if p.exists():\n                table=pd.read_csv(p,keep_default_na=False)\n                if not table.filename.is_unique:raise ValueError(\'Duplicate original transcript keys.\')\n                for r in table.to_dict(\'records\'):originals[(split,r[\'filename\'])]=(r.get(\'text\',r.get(\'transcript\',\'\')),str(p))\n        missing=[(r[\'split\'],r[\'filename\']) for r in self.jobs if (r[\'split\'],r[\'filename\']) not in sources]\n        if missing:raise ValueError(f\'{len(missing)} existing Crisper records missing. Set CRISPER_TRANSCRIPT_ROOT or attach Crisper outputs; automatic transcription is disabled. Examples: {missing[:5]}\')\n        reused=0;unavailable=0\n        for i,job in enumerate(self.jobs):\n            r,path=sources[(job[\'split\'],job[\'filename\'])];g=self.record(i)\n            if r.get(\'audio_sha256\')!=job[\'sha\']:raise ValueError(\'Crisper transcript WAV hash mismatch: \'+job[\'filename\'])\n            if r.get(\'model_id\')!=MODELS[\'asr\'] or r.get(\'model_revision\')!=\'f4334f6e8193f2691212d49b20fa12d370e13896\':raise ValueError(\'Unexpected Crisper source model/revision.\')\n            good=not r.get(\'error\') and r.get(\'mode\')==\'verbatim\' and bool(r.get(\'transcript\',\'\').strip())\n            text=r.get(\'transcript\',\'\') if good else \'\'\n            if good:reused+=1\n            else:unavailable+=1\n            old=g[\'verbatim\'].asstr()[()] if \'verbatim\' in g else None\n            if old is not None and old!=text:\n                for stage in [\'fluent\',\'align\',\'grammar\',\'text\',\'dsp\',\'wavlm\']:g.attrs[stage+\'_done\']=False\n                for name in [\'grammar_accuracy\',\'syntax_complexity\',\'repair_behaviour\',\'grammar_timing\',\'fluent_semantic\',\'fluent_surprisal\',\'word_timing\']:\n                    self.fixed(i,name,np.full(self.channels[name][\'dimensions\'],np.nan))\n            self.put(g,\'verbatim\',text);self.put(g,\'crisper_transcript\',text)\n            asrwords=r.get(\'words\',[]) if \'words\' in r else json.loads(r.get(\'words_json\') or \'[]\')\n            self.put(g,\'asr_words_json\',json.dumps(asrwords))\n            g.attrs.update(asr_done=True,asr_source=path,asr_estimated=True,\n                transcript_available=good,transcript_source_error=r.get(\'error\',\'\'),asr_inference_disabled=True)\n            original=originals.get((job[\'split\'],job[\'filename\']))\n            if original:\n                self.put(g,\'original_transcript\',original[0]);g.attrs[\'original_transcript_source\']=original[1]\n                g.attrs[\'original_transcript_join\']=\'split+filename from legacy export; no per-WAV hash in source\'\n        self.h.flush()\n        print(f\'Existing transcripts only: {reused} usable Crisper records; {unavailable} unavailable. Original transcripts retained when present. No ASR weights downloaded or transcription performed.\',flush=True)\n\n    def fluent(self):\n        """Compatibility stage: reuse existing text verbatim, with no cleanup model."""\n        method=\'Existing verbatim transcript unchanged; Qwen cleanup disabled\'\n        for i in range(len(self.jobs)):\n            if not self.done(i,\'asr\'):continue\n            g=self.record(i)\n            if g.attrs.get(\'fluent_method\')==method and self.done(i,\'fluent\'):continue\n            text=g[\'verbatim\'].asstr()[()]\n            # Text-derived checkpoints from cleaned text must be recomputed.\n            for stage in [\'grammar\',\'text\']:g.attrs[stage+\'_done\']=False\n            for name in [\'grammar_accuracy\',\'syntax_complexity\',\'repair_behaviour\',\n                         \'grammar_timing\',\'fluent_semantic\',\'fluent_surprisal\']:\n                self.fixed(i,name,np.full(self.channels[name][\'dimensions\'],np.nan))\n            self.put(g,\'fluent\',text)\n            self.put(g,\'fluent_to_verbatim_token\',np.arange(len(text.split()),dtype=\'i4\'))\n            self.put(g,\'verbatim_tokens_json\',json.dumps(text.split()))\n            self.put(g,\'repair_events_json\',\'[]\')\n            g.attrs.update(fluent_done=True,fluent_method=method,\n                           repair_events_available=False,qwen_cleanup_disabled=True)\n            if \'fluent_error\' in g.attrs:del g.attrs[\'fluent_error\']\n        self.h.attrs[\'qwen_cleanup_disabled\']=True\n        self.h.attrs[\'text_feature_source\']=\'Existing verbatim transcript unchanged\'\n        self.h.flush()\n        print(\'Qwen disabled: existing transcripts used unchanged. Repair features unavailable.\',flush=True)\n\n    def align(self):\n        pending=[i for i in self.pending(\'align\') if self.done(i,\'asr\')]\n        if not pending:return\n        import whisperx\n        from difflib import SequenceMatcher\n        model,metadata=whisperx.load_align_model(language_code=\'en\',device=self.device,\n            model_name=MODELS[\'align\'])\n        def one(i,g):\n            text=g[\'verbatim\'].asstr()[()]\n            if not text.strip(): aligned=[]\n            else:\n                result=whisperx.align([dict(start=0.,end=self.jobs[i][\'duration\'],text=text)],\n                    model,metadata,self.audio(i),self.device,return_char_alignments=False)\n                aligned=result.get(\'word_segments\',[])\n            self.put(g,\'aligned_words_json\',json.dumps(aligned))\n            # Map original whitespace-token positions to aligner\'s words without inventing timings.\n            normalize=lambda s:re.sub(r\'\\W\',\'\',s.lower())\n            a=[normalize(t) for t in text.split()]; b=[normalize(w[\'word\']) for w in aligned]\n            positions=np.full((len(a),2),np.nan,dtype=\'f4\')\n            for block in SequenceMatcher(None,a,b,autojunk=False).get_matching_blocks():\n                for k in range(block.size):\n                    w=aligned[block.b+k]\n                    if \'start\' in w and \'end\' in w: positions[block.a+k]=[w[\'start\'],w[\'end\']]\n            self.put(g,\'verbatim_token_times\',positions)\n            g.attrs[\'alignment_method\']=\'WhisperX character-CTC forced alignment; not phoneme posterior scores\'\n        self.each(\'align\',pending,one); del model; self.release()\n\n    def grammar(self):\n        pending=[i for i in self.pending(\'grammar\') if self.done(i,\'fluent\') and self.done(i,\'align\')]\n        if not pending:return\n        import spacy,errant\n        from transformers import AutoTokenizer,AutoModelForSeq2SeqLM\n        nlp=spacy.load(\'en_core_web_sm\',disable=[\'ner\']); annotator=errant.load(\'en\',nlp=nlp)\n        rev=self.revision(\'gec\');tok=AutoTokenizer.from_pretrained(MODELS[\'gec\'],revision=rev)\n        model=AutoModelForSeq2SeqLM.from_pretrained(MODELS[\'gec\'],revision=rev).to(self.device).eval()\n        self.h[\'provenance\'].attrs[\'spacy_model\']=json.dumps(nlp.meta)\n        def generate_corrections(batch):\n            inp=tok([\'grammar: \'+s for s in batch],padding=True,return_tensors=\'pt\',truncation=False).to(self.device)\n            if inp.input_ids.shape[1]>512:raise ValueError(\'GEC span exceeds token budget; no silent truncation.\')\n            with torch.inference_mode():generated=model.generate(**inp,max_new_tokens=384,num_beams=4,do_sample=False)\n            return tok.batch_decode(generated,skip_special_tokens=True)\n        def one(i,g):\n            text=g[\'fluent\'].asstr()[()];doc=nlp(text); n=len(words(text))\n            if not n:\n                for name in [\'grammar_accuracy\',\'syntax_complexity\',\'repair_behaviour\',\'grammar_timing\']:\n                    self.fixed(i,name,np.full(self.channels[name][\'dimensions\'],np.nan))\n                self.put(g,\'corrected_proposal\',\'\');self.put(g,\'grammar_edits_json\',\'[]\');return\n            sentlist=list(doc.sents); edits=[];proposals=[]\n            inputs=[]\n            for sent in sentlist:\n                tokens=list(sent)\n                for a in range(0,len(tokens),80):\n                    chunk=tokens[a:a+80];inputs.append(doc[chunk[0].i:chunk[-1].i+1].text)\n            corrections={s:c for s,c in gpu_batch_map(inputs,min(16,self.gpu_batch),generate_corrections)}\n            # Split unusually long ASR sentences without silently truncating GEC input.\n            for sent in sentlist:\n                tokens=list(sent); chunks=[tokens[a:a+80] for a in range(0,len(tokens),80)]\n                for chunk in chunks:\n                    original=doc[chunk[0].i:chunk[-1].i+1].text\n                    corrected=corrections[original]; proposals.append(corrected)\n                    original_doc=annotator.parse(original);corrected_doc=annotator.parse(corrected)\n                    # ERRANT uses spaCy tokenization; map local offsets by character spans.\n                    base=chunk[0].idx\n                    for e in annotator.annotate(original_doc,corrected_doc):\n                        if e.o_start<len(original_doc):char=base+original_doc[e.o_start].idx\n                        else:char=base+len(original)\n                        global_start=next((t.i for t in doc if t.idx>=char),len(doc))\n                        endchar=base+(original_doc[e.o_end-1].idx+len(original_doc[e.o_end-1]) if e.o_end>e.o_start else char-base)\n                        global_end=next((t.i for t in doc if t.idx>=endchar),len(doc))\n                        edits.append(dict(type=e.type,original=e.o_str,correction=e.c_str,start=global_start,end=global_end))\n            self.put(g,\'corrected_proposal\',\' \'.join(proposals));self.put(g,\'grammar_edits_json\',json.dumps(edits))\n            # Explicit clause proxy: verb heads of roots and clausal dependencies.\n            clause_deps={\'ROOT\',\'advcl\',\'ccomp\',\'xcomp\',\'relcl\',\'acl\',\'conj\',\'parataxis\'}\n            heads=[t for t in doc if t.pos_ in {\'VERB\',\'AUX\'} and t.dep_ in clause_deps]\n            def owner(token):\n                chain=[token]+list(token.ancestors)\n                return next((t.i for t in chain if t in heads),None)\n            owners=[owner(t) for t in doc]\n            counts={h.i:sum(v==h.i for v in owners) for h in heads}\n            edit_counts={h.i:0 for h in heads};sentence_errors=[0]*len(sentlist)\n            for e in edits:\n                k=min(e[\'start\'],len(doc)-1); o=owners[k]\n                if o is not None:edit_counts[o]+=1\n                for j,s in enumerate(sentlist):\n                    if s.start<=k<s.end:sentence_errors[j]+=1;break\n            kinds=[e[\'type\'] for e in edits]\n            count=lambda suffix:sum(k.endswith(suffix) for k in kinds)\n            severe=sum(k.endswith((\'VERB:SVA\',\'VERB:FORM\',\'WO\')) or k in {\'M:VERB\',\'M:AUX\'} for k in kinds)\n            negation=sum(bool(set(words(e[\'original\'].lower())+words(e[\'correction\'].lower())) & {\'not\',\'never\',\'no\',"n\'t"}) for e in edits)\n            accuracy=[rate(len(edits),n),rate(severe,n),ratio(sum(v==0 for v in edit_counts.values()),len(heads)),\n                ratio(sum(v==0 for v in sentence_errors),len(sentlist)),rate(count(\'VERB:TENSE\'),n),\n                rate(count(\'VERB:SVA\'),n),rate(count(\'VERB:FORM\'),n),rate(count(\'DET\'),n),rate(count(\'PREP\'),n),\n                rate(count(\'PRON\'),n),rate(count(\'NOUN:NUM\'),n),rate(count(\'WO\'),n),\n                rate(sum(k.startswith(\'M:\') for k in kinds),n),rate(sum(k.startswith(\'U:\') for k in kinds),n),\n                rate(negation,n),ratio(sum(v>=2 for v in edit_counts.values()),len(heads))]\n            depths=np.asarray([len(list(t.ancestors)) for t in doc if not t.is_punct])\n            complex_heads=[h for h in heads if h.dep_ in {\'advcl\',\'ccomp\',\'xcomp\',\'relcl\',\'acl\'}]\n            complete=[h for h in heads if any(c.dep_ in {\'nsubj\',\'nsubjpass\',\'csubj\'} for c in h.children)\n                or (h.dep_==\'ROOT\' and h.tag_==\'VB\')]\n            syntax=[ratio(sum(counts.values()),len(heads)),ratio(len(heads),len(sentlist)),\n                ratio(len(complex_heads),len(heads)),rate(sum(h.dep_==\'relcl\' for h in heads),n),\n                rate(sum(t.lower_ in {\'if\',\'unless\'} and t.dep_==\'mark\' for t in doc),n),\n                rate(sum(t.dep_==\'auxpass\' for t in doc),n),ratio(sum(any(t.dep_ in {\'advcl\',\'ccomp\',\'relcl\'} for t in s) for s in sentlist),len(sentlist)),\n                float(depths.mean()) if len(depths) else np.nan,float(np.quantile(depths,.9)) if len(depths) else np.nan,\n                ratio(len(complete),len(heads)),rate(sum(not any(t in heads for t in s) for s in sentlist),n),\n                ratio(sum(edit_counts[h.i]==0 for h in complex_heads),len(complex_heads))]\n            events=self.getjson(g,\'repair_events_json\',[]); nw=len(words(g[\'verbatim\'].asstr()[()]))\n            repairs=[e for e in events if e[\'type\']==\'grammar_repair\']; evaluated=[e for e in repairs if type(e.get(\'success\'))==bool]\n            restarts=[e for e in events if e[\'type\']==\'false_start\']\n            repairs_v=[rate(len(repairs),nw),rate(sum(e.get(\'success\') is True for e in repairs),nw),\n                rate(sum(e.get(\'success\') is False for e in repairs),nw),\n                ratio(sum(e.get(\'success\') is True for e in evaluated),len(evaluated)),\n                rate(len(restarts),nw),rate(sum(e[\'type\']==\'abandoned_clause\' for e in events),nw),\n                rate(sum(e[\'type\']==\'repetition\' for e in events),nw),\n                ratio(sum(e.get(\'completed\') is True for e in restarts),sum(type(e.get(\'completed\'))==bool for e in restarts))]\n            if not g.attrs.get(\'repair_events_available\',False):repairs_v=[np.nan]*8\n            # Timing on existing transcript tokens; clause boundaries are parser estimates.\n            token_times=g[\'verbatim_token_times\'][:]; retained=g[\'fluent_to_verbatim_token\'][:]\n            fluent_times=token_times[retained]; self.put(g,\'fluent_token_times\',fluent_times)\n            spans=list(re.finditer(r\'\\S+\',text)); boundary_positions=[]\n            for j,span in enumerate(spans):\n                t=next((t for t in doc if t.idx>=span.start()),None)\n                if t is not None and (t.is_sent_start or t.dep_==\'mark\' or (t in heads and t.dep_!=\'ROOT\')):boundary_positions.append(j)\n            gaps=[]; inside=[];boundaries=[]; speeds=[];runs=[];run=0\n            for j in range(len(fluent_times)):\n                a,b=fluent_times[j]\n                if np.isfinite(a+b) and b>a:speeds.append(1/(b-a))\n                run+=1\n                if j and np.isfinite(fluent_times[j-1,1]+a):\n                    gap=max(0.,a-fluent_times[j-1,1])\n                    if gap>=.25:\n                        gaps.append(gap);is_boundary=j in boundary_positions\n                        boundaries.append(is_boundary)\n                        if not is_boundary:inside.append(gap)\n                        runs.append(run-1);run=1\n            if run:runs.append(run)\n            before=[]\n            for e in repairs:\n                ids=e.get(\'replacement_indices\',[])\n                if ids:\n                    j=min(ids)\n                    if j and j<len(token_times) and np.isfinite(token_times[j,0]+token_times[j-1,1]):\n                        before.append(max(0.,token_times[j,0]-token_times[j-1,1]))\n            timing=[rate(len(inside),n),ratio(sum(boundaries),len(boundaries)),float(np.median(before)) if before else np.nan,\n                rate(sum(p>1 for p in inside),n),float(np.mean(runs)) if runs else np.nan,\n                ratio(np.std(speeds),np.mean(speeds)) if speeds else np.nan]\n            if not np.isfinite(fluent_times).any():timing=[np.nan]*6\n            for name,value in [(\'grammar_accuracy\',accuracy),(\'syntax_complexity\',syntax),(\'repair_behaviour\',repairs_v),(\'grammar_timing\',timing)]:self.fixed(i,name,value)\n            self.put(g,\'clause_analysis_json\',json.dumps(dict(head_indices=[h.i for h in heads],word_counts=counts,\n                grammatical_edit_counts=edit_counts,complex_head_indices=[h.i for h in complex_heads])))\n            g.attrs[\'grammar_method\']=\'GEC+ERRANT proposals; DET includes determiners; severity/negation/clause completeness heuristic; not human grammar annotation\'\n        self.each(\'grammar\',pending,one);del model,tok,nlp,annotator;self.release()\n\n    def text(self):\n        pending=[i for i in self.pending(\'text\') if self.done(i,\'fluent\')]\n        if not pending:return\n        from transformers import AutoTokenizer,AutoModel,AutoModelForCausalLM\n        rev=self.revision(\'semantic\');tok=AutoTokenizer.from_pretrained(MODELS[\'semantic\'],revision=rev)\n        model=AutoModel.from_pretrained(MODELS[\'semantic\'],revision=rev).to(self.device).eval()\n        def semantic(i,g):\n            ids=tok(g[\'fluent\'].asstr()[()],add_special_tokens=False)[\'input_ids\']; sums=[];weights=[]\n            for start in range(0,len(ids),254):\n                part=ids[start:start+254];batch=tok.prepare_for_model(part,return_tensors=\'pt\')\n                batch={k:v.unsqueeze(0).to(self.device) for k,v in batch.items()}\n                with torch.inference_mode():states=model(**batch).last_hidden_state[0][1:-1]\n                sums.append(states.sum(0).float().cpu().numpy());weights.append(len(states))\n            v=sum(sums)/sum(weights) if weights else np.full(384,np.nan)\n            if weights:v=v/max(float(np.linalg.norm(v)),1e-12)\n            self.fixed(i,\'fluent_semantic\',v)\n        # Do not mark text done until both text channels are written.\n        successful=[]\n        for i in tqdm(pending,desc=\'semantic\'):\n            try:semantic(i,self.record(i));successful.append(i)\n            except Exception as e:self.record(i).attrs[\'text_error\']=str(e)\n            self.drain_dsp()\n            self.h.flush()\n        del model,tok;self.release()\n        rev=self.revision(\'surprisal\');tok=AutoTokenizer.from_pretrained(MODELS[\'surprisal\'],revision=rev)\n        model=AutoModelForCausalLM.from_pretrained(MODELS[\'surprisal\'],revision=rev).to(self.device).eval()\n        def surprise(i,g):\n            ids=tok(g[\'fluent\'].asstr()[()],add_special_tokens=False)[\'input_ids\']; losses=[]\n            # Each target token scored once with up to 512 preceding tokens; no suffix truncation.\n            for start in range(0,len(ids),256):\n                target=ids[start:start+256];context=ids[max(0,start-512):start]\n                seq=[tok.bos_token_id]+context+target; inp=torch.tensor([seq],device=self.device)\n                with torch.inference_mode():logits=model(inp).logits[0]\n                lp=torch.log_softmax(logits[:-1].float(),-1)\n                nll=-lp.gather(1,inp[0,1:,None]).squeeze(1)\n                losses.extend(nll[-len(target):].cpu().numpy())\n            self.fixed(i,\'fluent_surprisal\',stats(losses))\n        self.each(\'text\',successful,surprise);del model,tok;self.release()\n\n    def dsp(self,background=False):\n        import multiprocessing\n        pending=[i for i in self.pending(\'dsp\') if self.done(i,\'align\')]\n        if not pending:return\n        tasks=[(i,self.jobs[i][\'path\'],self.getjson(self.record(i),\'aligned_words_json\',[])) for i in pending]\n        self._dsp_pool=ProcessPoolExecutor(max_workers=min(self.cpu_workers,len(tasks)),\n            mp_context=multiprocessing.get_context(\'spawn\'))\n        self._dsp_futures={self._dsp_pool.submit(dsp_worker,task):task[0] for task in tasks}\n        self._dsp_bar=tqdm(total=len(tasks),desc=f\'DSP ({self.cpu_workers} CPU workers; overlaps GPU)\')\n        if not background:self.drain_dsp(wait=True)\n\n    def drain_dsp(self,wait=False):\n        futures=getattr(self,\'_dsp_futures\',{})\n        completed=as_completed(list(futures)) if wait else [f for f in futures if f.done()]\n        for future in completed:\n            i=futures.pop(future);g=self.record(i)\n            try:\n                _,vectors,metadata=future.result()\n                for name,value in vectors.items():self.fixed(i,name,value)\n                g.attrs[\'acoustic_rates_json\']=json.dumps(metadata,allow_nan=False)\n                g.attrs[\'dsp_done\']=True\n                if \'dsp_error\' in g.attrs:del g.attrs[\'dsp_error\']\n            except Exception as e:\n                g.attrs[\'dsp_error\']=str(e)[:2000];tqdm.write(f\'DSP row {i} failed: {e}\')\n            self._dsp_bar.update(1);self.h.flush()\n        if hasattr(self,\'_dsp_pool\') and not futures:\n            self._dsp_pool.shutdown(wait=True);self._dsp_bar.close();del self._dsp_pool\n\n    def ranges(self,length):\n        size=16000*10; ranges=[(a,min(a+size,length)) for a in range(0,length,size)]\n        if len(ranges)>1 and ranges[-1][1]-ranges[-1][0]<1024:\n            ranges[-2]=(ranges[-2][0],length);ranges.pop()\n        return ranges\n\n    @staticmethod\n    def centers(model,frames,start):\n        cfg=model.config;jump=1;field=1\n        for kernel,stride in zip(cfg.conv_kernel,cfg.conv_stride):field+=(kernel-1)*jump;jump*=stride\n        return (start+np.arange(frames)*jump+(field-1)/2)/16000\n\n    def speech(self,kind):\n        pending=self.pending(kind)\n        if kind==\'wavlm\':pending=[i for i in pending if self.done(i,\'align\')]\n        if not pending:return\n        from transformers import AutoFeatureExtractor,WavLMModel,HubertModel,Wav2Vec2Model\n        rev=self.revision(kind);processor=AutoFeatureExtractor.from_pretrained(MODELS[kind],revision=rev)\n        cls={\'wavlm\':WavLMModel,\'hubert\':HubertModel,\'wav2vec2\':Wav2Vec2Model}[kind]\n        model=cls.from_pretrained(MODELS[kind],revision=rev).to(self.device).eval()\n        def one(i,g):\n            x=self.audio(i);layers=[4,8,12] if kind==\'wavlm\' else [4,12]\n            sums={l:np.zeros(768,np.float64) for l in layers};squares={l:np.zeros(768,np.float64) for l in layers};count=0\n            frames8=[];times=[];chunks=[];bounds=[]\n            segments=self.ranges(len(x));groups={};encoded={}\n            # Equal-length groups avoid padded frames and padding-dependent encoder changes.\n            for a,b in segments:groups.setdefault(b-a,[]).append((a,b))\n            def encode(batch):\n                inputs=processor([x[a:b] for a,b in batch],sampling_rate=16000,return_tensors=\'pt\',padding=False).to(self.device)\n                with torch.inference_mode(),torch.autocast(\'cuda\',dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):\n                    output=model(**inputs,output_hidden_states=True)\n                return [{l:output.hidden_states[l][j].float().cpu().numpy() for l in layers} for j in range(len(batch))]\n            for group in groups.values():\n                for segment,result in gpu_batch_map(group,self.gpu_batch,encode):encoded[segment[0]]=result\n            for a,b in segments:\n                for l in layers:\n                    f=encoded[a][l]\n                    sums[l]+=f.astype(\'f8\').sum(0);squares[l]+=(f.astype(\'f8\')**2).sum(0)\n                count+=len(f)\n                if kind==\'wavlm\':\n                    f8=encoded[a][8]\n                    frames8.append(f8);times.append(self.centers(model,len(f8),a))\n                    chunks.append(f8.mean(0));bounds.append([a/16000,b/16000])\n                del encoded[a]\n            for l in layers:\n                mean=sums[l]/count;std=np.sqrt(np.maximum(squares[l]/count-mean**2,0))\n                self.fixed(i,f\'{kind}_layer{l}_mean\',mean)\n                if kind==\'wavlm\' or l==12:self.fixed(i,f\'{kind}_layer{l}_std\',std)\n            if kind==\'wavlm\':\n                frames=np.concatenate(frames8); t=np.concatenate(times)\n                aligned=self.getjson(g,\'aligned_words_json\',[]); W=np.full((len(aligned),768),np.nan,np.float32)\n                word_bounds=np.full((len(aligned),2),np.nan,np.float32)\n                for j,w in enumerate(aligned):\n                    if \'start\' not in w or \'end\' not in w:continue\n                    a,b=float(w[\'start\']),float(w[\'end\']);word_bounds[j]=[a,b]\n                    mask=(t>=a)&(t<b)\n                    if mask.any():W[j]=frames[mask].mean(0)\n                self.put(g,\'word_wavlm\',W);self.put(g,\'word_wavlm_available\',np.isfinite(W).all(1))\n                self.put(g,\'word_times\',word_bounds);self.put(g,\'chunk_wavlm\',np.asarray(chunks,\'f4\'))\n                self.put(g,\'chunk_times\',np.asarray(bounds,\'f4\'))\n                g.attrs[\'word_embedding_layer\']=8;g.attrs[\'chunk_embedding_layer\']=8\n                g.attrs[\'pooling\']=\'Global frame-weighted population moments across unpadded 10-second contexts\'\n            g.attrs[kind+\'_precision\']=\'bfloat16 CUDA autocast\' if torch.cuda.is_bf16_supported() else \'float16 CUDA autocast\'\n        self.each(kind,pending,one);del model,processor;self.release()\n\n    def phoneme(self):\n        pending=self.pending(\'phoneme\')\n        if not pending:return\n        from transformers import AutoFeatureExtractor,Wav2Vec2ForCTC\n        from huggingface_hub import hf_hub_download\n        rev=self.revision(\'phoneme\');processor=AutoFeatureExtractor.from_pretrained(MODELS[\'phoneme\'],revision=rev)\n        model=Wav2Vec2ForCTC.from_pretrained(MODELS[\'phoneme\'],revision=rev).to(self.device).eval()\n        vocab=json.loads(Path(hf_hub_download(MODELS[\'phoneme\'],\'vocab.json\',revision=rev)).read_text())\n        self.h[\'provenance\'][\'phoneme\'].attrs[\'vocabulary_json\']=json.dumps(vocab)\n        if model.config.vocab_size!=392:raise ValueError(\'Phoneme vocabulary width changed.\')\n        def one(i,g):\n            x=self.audio(i);rows=[];times=[]\n            segments=self.ranges(len(x));groups={};encoded={}\n            for a,b in segments:groups.setdefault(b-a,[]).append((a,b))\n            def encode(batch):\n                inputs=processor([x[a:b] for a,b in batch],sampling_rate=16000,return_tensors=\'pt\',padding=False).to(self.device)\n                with torch.inference_mode(),torch.autocast(\'cuda\',dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):\n                    logits=model(**inputs).logits\n                probabilities=logits.float().softmax(-1).cpu().numpy().astype(\'f2\')\n                return [probabilities[j] for j in range(len(batch))]\n            for group in groups.values():\n                for segment,result in gpu_batch_map(group,self.gpu_batch,encode):encoded[segment[0]]=result\n            for a,b in segments:\n                probabilities=encoded.pop(a);rows.append(probabilities)\n                times.append(self.centers(model,len(probabilities),a))\n            self.put(g,\'phoneme_probabilities\',np.concatenate(rows));self.put(g,\'phoneme_frame_times\',np.concatenate(times).astype(\'f4\'))\n            g.attrs[\'phoneme_note\']=\'392 CTC symbols including blanks/special tokens; frame posteriors, not calibrated pronunciation or grammar scores\'\n        self.each(\'phoneme\',pending,one);del model,processor;self.release()\n\n    def finish(self):\n        failures={s:[i for i in range(len(self.jobs)) if not self.done(i,s)] for s in STAGES}\n        complete=all(not v for v in failures.values()) and len(self.jobs)==985\n        self.h.attrs.update(complete=complete,processing_status_json=json.dumps(failures),\n            completed_at_utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),\n            missing_fixed_values=int((~self.h[\'available\'][:]).sum()))\n        self.h.flush();self.h.close()\n        print(\'COMPLETE\' if complete else \'PARTIAL: resume to retry missing stages\',self.path)\n        print(\'Only one output artifact. Missing/uncertain features stay NaN with availability masks.\')\n        if not complete:print(\'Unfinished row indices by stage:\',{k:v for k,v in failures.items() if v})\n\n    def run(self):\n        try:\n            self.transcribe();self.fluent();self.align();self.dsp(background=True);self.grammar();self.text()\n            for kind in [\'wavlm\',\'hubert\',\'wav2vec2\']:self.speech(kind)\n            self.phoneme();self.drain_dsp(wait=True);self.finish()\n        finally:\n            if hasattr(self,\'_dsp_pool\'):\n                for process in (getattr(self._dsp_pool,\'_processes\',{}) or {}).values():\n                    if process.is_alive():process.terminate()\n                self._dsp_pool.shutdown(wait=False,cancel_futures=True)\n                self._dsp_bar.close()\n            if self.h.id.valid:self.h.flush();self.h.close()\n'
import sys
RUNTIME_FILE = Path("/tmp/shl_channel_runtime.py")
sys.path.insert(0, str(RUNTIME_FILE.parent))
RUNTIME_FILE.write_text(RUNTIME_SOURCE)
import importlib.util
spec = importlib.util.spec_from_file_location("shl_channel_runtime", RUNTIME_FILE)
runtime = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runtime
spec.loader.exec_module(runtime)
print("Ready: 26 fixed channels + 3 sequence channels; 9,888 fixed dimensions.")


## Extract all recordings

1. Each stage displays progress and records per-row errors inside the file. Successful stages are checkpointed immediately.
2. GPU work uses batched requests and equal-length audio chunks, with automatic OOM batch splitting. CPU DSP uses a spawned process pool concurrently with GPU grammar/embedding work, with one parent HDF5 writer. Qwen is removed. Grammar correction and text analysis use existing transcripts; no ASR or supervised training occurs.
3. A finished processing pass can still contain missing/uncertain individual features. Inspect masks and stage errors before downstream training.

In [ ]:
extractor = runtime.Extractor(CONFIG)
extractor.run()

## Inspect and download the single output

1. Use Kaggle's output pane or the link below to download the single `.h5` file. Save a version with outputs to retain it.
2. This inspection shows processing completion, missing values, and an example row. It does not fit or validate a predictive model.

In [ ]:
import h5py, json
from IPython.display import display, FileLink
with h5py.File(CONFIG["OUTPUT"], "r") as f:
    print("Complete processing:", bool(f.attrs["complete"]))
    print("Fixed matrix:", f["X"].shape)
    print("Training rows:", len(f["metadata/train_indices"]))
    print("Test rows:", len(f["metadata/test_indices"]))
    print("Unavailable fixed entries:", f.attrs["missing_fixed_values"])
    print("Unfinished stages:", {k:v for k,v in json.loads(f.attrs["processing_status_json"]).items() if v})
    if "0000" in f["records"]:
        r = f["records/0000"]
        for key in ["word_wavlm", "phoneme_probabilities", "chunk_wavlm"]:
            print(key, r[key].shape if key in r else "unavailable; inspect stage error")
display(FileLink(CONFIG["OUTPUT"]))

## Later training: reading one row

1. The following optional cell only illustrates reading the artifact. Keep its separate channels available rather than treating all 9,888 dimensions as equally useful.
2. Train using the original training labels. Fit imputation, scaling, dimensionality reduction and learned sequence pooling inside each validation fold. Build the stacker from out-of-fold predictions. Test labels remain unknown.
3. Dependency-clause segmentation, article/determiner edits, severity, negation and repair success use documented proxies. In particular, reported error-free clauses mean **no proposed edit was assigned**, not proven error-free speech. Evaluate against manually inspected examples before claiming grammatical validity.

In [ ]:
# Example reader; does not train anything.
with h5py.File(CONFIG["OUTPUT"], "r") as f:
    i = 0
    row = f["X"][i]
    availability = f["available"][i]
    channels = json.loads(f.attrs["channels_json"])
    separate_vectors = {name: row[d["start"]:d["stop"]] for name,d in channels.items()}
    print("Channel names:", list(separate_vectors))
    print("WavLM multilayer shape:", sum(len(v) for k,v in separate_vectors.items() if k.startswith("wavlm_")))